# Reproducing Belief State Geometry Experiments (Mess3 + RRXOR)

**Paper**: "Transformers Represent Belief State Geometry in their Residual Stream"
(Shai, Marzen, Teixeira, Oldenziel, Riechers — NeurIPS 2024, arXiv:2405.15943v3)

**Runtime**: Use a **GPU runtime** (Runtime > Change runtime type > T4 GPU).
Full 1M-step training takes ~20-40 min per model on a T4.

This notebook is **self-contained** — no clone or local install needed. It trains
both models from the paper and saves checkpoints for future analysis:

**Part 1 — Mess3** (Figures 5, 6): Fractal belief geometry in the final residual stream.
**Part 2 — RRXOR** (Figure 7): Belief geometry spread across multiple layers.
**Part 3 — Save**: Download trained weights for both models.

In [ ]:
!pip install -q transformer-lens 2>&1 | tail -1
import torch, sys
print(f'Python {sys.version.split()[0]}  |  PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import itertools, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformer_lens import HookedTransformer, HookedTransformerConfig

warnings.filterwarnings('ignore')

# ── Mess3 process (Appendix A.3) ──
MESS3_X = 0.05
MESS3_A = 0.85

# ── Transformer architecture (Appendix A.6) ──
N_CTX = 10
D_MODEL = 64
D_HEAD = 8
N_HEADS = 1
N_LAYERS = 4
D_MLP = 256
VOCAB_SIZE = 3

# ── Training (Appendix A.6) ──
BATCH_SIZE = 64
LEARNING_RATE = 0.01
SEQUENCE_LEN = N_CTX + 1

# Paper uses 1,000,000 steps. Feasible on a T4 GPU (~20-40 min).
NUM_STEPS = 1_000_000

# ── Analysis ──
MSP_DEPTH = 8
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}  |  Steps: {NUM_STEPS:,}')

---
# Part 1: Mess3 (Figures 5, 6)

## 1. Define and Verify the Mess3 Process (Appendix A.3)

In [ ]:
def mess3(x, a):
    """Mess3 transition matrices (Appendix A.3)."""
    b, y = (1 - a) / 2, 1 - 2 * x
    ay, bx, by, ax = a*y, b*x, b*y, a*x
    return np.array([
        [[ay,bx,bx],[ax,by,bx],[ax,bx,by]],
        [[by,ax,bx],[bx,ay,bx],[bx,ax,by]],
        [[by,bx,ax],[bx,by,ax],[bx,bx,ay]],
    ], dtype=np.float64)

tmats = mess3(MESS3_X, MESS3_A)

# ── Verify against paper's exact matrices ──
expected = {
    'A': np.array([[0.765,0.00375,0.00375],[0.0425,0.0675,0.00375],[0.0425,0.00375,0.0675]]),
    'B': np.array([[0.0675,0.0425,0.00375],[0.00375,0.765,0.00375],[0.00375,0.0425,0.0675]]),
    'C': np.array([[0.0675,0.00375,0.0425],[0.00375,0.0675,0.0425],[0.00375,0.00375,0.765]]),
}
for i, name in enumerate('ABC'):
    assert np.allclose(tmats[i], expected[name], atol=1e-10), f'T^({name}) mismatch'
assert np.allclose(tmats.sum(axis=0).sum(axis=1), 1.0, atol=1e-10), 'Rows do not sum to 1'

# ── Stationary distribution (left eigenvector of T) ──
T_net = tmats.sum(axis=0)
eigvals, eigvecs = np.linalg.eig(T_net.T)
stationary = eigvecs[:, np.isclose(eigvals, 1)].real.squeeze()
stationary = stationary / stationary.sum()

print('PASS: All transition matrices match paper Appendix A.3.')
print(f'Stationary distribution: {stationary}')
for i, name in enumerate('ABC'):
    print(f'T^({name}) =\\n{tmats[i]}\\n')

## 2. Ground-Truth Belief State Geometry (Mixed-State Presentation)

Each belief state is a probability distribution over 3 hidden states (a point in
the 2-simplex). The MSP of Mess3 has a **fractal** structure.

In [ ]:
def build_msp_beliefs(tmats, init_state, max_depth):
    """Build MSP tree by breadth-first expansion; return all belief states."""
    beliefs = [init_state.copy()]
    queue = [(init_state.copy(), 0)]
    while queue:
        state, depth = queue.pop(0)
        if depth >= max_depth:
            continue
        for x in range(tmats.shape[0]):
            raw = state @ tmats[x]
            norm = raw.sum()
            if norm > 1e-15:
                child = raw / norm
                beliefs.append(child)
                queue.append((child, depth + 1))
    return np.array(beliefs)

belief_states_gt = build_msp_beliefs(tmats, stationary, MSP_DEPTH)
print(f'MSP belief states: {len(belief_states_gt):,}')
assert np.allclose(belief_states_gt.sum(axis=1), 1.0, atol=1e-4), 'Belief states do not sum to 1'
print('PASS: All belief states sum to 1.')

# ── Simplex plotting helpers ──
def simplex_to_xy(bs):
    return bs[:, 1] + 0.5 * bs[:, 2], (np.sqrt(3) / 2) * bs[:, 2]

def plot_simplex(bs, title, ax, colors=None, s=1, alpha=0.5):
    x, y = simplex_to_xy(bs)
    c = np.clip(bs, 0, 1) if colors is None else np.clip(colors, 0, 1)
    ax.scatter(x, y, c=c, s=s, alpha=alpha, edgecolors='none')
    ax.add_patch(plt.Polygon([[0,0],[1,0],[0.5,np.sqrt(3)/2]], fill=False, ec='gray', lw=0.5))
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 0.95)
    ax.set_aspect('equal'); ax.set_title(title, fontsize=13); ax.axis('off')

fig, ax = plt.subplots(figsize=(6, 5.2))
plot_simplex(belief_states_gt, 'Ground-Truth Belief State Geometry (Mess3 MSP)', ax, s=2, alpha=0.6)
plt.tight_layout(); plt.show()

## 3. Initialize Transformer (Appendix A.6)

In [ ]:
torch.manual_seed(SEED)
try:
    tl_cfg = HookedTransformerConfig(
        n_ctx=N_CTX, d_model=D_MODEL, d_head=D_HEAD, n_heads=N_HEADS,
        n_layers=N_LAYERS, d_mlp=D_MLP, d_vocab=VOCAB_SIZE,
        act_fn='relu', normalization_type='LN', device=DEVICE, seed=SEED,
    )
    model = HookedTransformer(tl_cfg)
    print(f'Model: n_heads={N_HEADS}, d_head={D_HEAD}, d_model={D_MODEL}')
except Exception as e:
    print(f'Paper config failed ({e}); falling back to n_heads=8.')
    N_HEADS = 8
    tl_cfg = HookedTransformerConfig(
        n_ctx=N_CTX, d_model=D_MODEL, d_head=D_HEAD, n_heads=N_HEADS,
        n_layers=N_LAYERS, d_mlp=D_MLP, d_vocab=VOCAB_SIZE,
        act_fn='relu', normalization_type='LN', device=DEVICE, seed=SEED,
    )
    model = HookedTransformer(tl_cfg)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}  |  Device: {DEVICE}')

## 4. Train the Transformer

SGD, lr=0.01, batch size 64, 1,000,000 steps (Appendix A.6).
Each batch is freshly sampled from the stationary distribution.

In [ ]:
# ── Data generation from the HMM ──
def generate_batch(rng):
    """Generate one batch of sequences from the Mess3 HMM."""
    tokens = np.zeros((BATCH_SIZE, SEQUENCE_LEN), dtype=np.int64)
    for i in range(BATCH_SIZE):
        state = stationary.copy()
        for t in range(SEQUENCE_LEN):
            obs_probs = np.array([np.sum(state @ tmats[x]) for x in range(VOCAB_SIZE)])
            obs_probs /= obs_probs.sum()
            tok = rng.choice(VOCAB_SIZE, p=obs_probs)
            tokens[i, t] = tok
            state = state @ tmats[tok]
            state /= state.sum()
    inp = torch.tensor(tokens[:, :-1], device=DEVICE)
    lab = torch.tensor(tokens[:, 1:], device=DEVICE, dtype=torch.long)
    return inp, lab

# ── Precompute exhaustive analysis data (paper: "all possible input sequences") ──
print('Enumerating all sequences...')
all_seqs_np = np.array(list(itertools.product(range(VOCAB_SIZE), repeat=N_CTX)), dtype=np.int32)
all_seqs_torch = torch.tensor(all_seqs_np, dtype=torch.long)
n_all_seqs = all_seqs_np.shape[0]

print('Computing belief states...')
n_states = tmats.shape[1]
all_beliefs = np.zeros((n_all_seqs, N_CTX, n_states), dtype=np.float32)
for i in range(n_all_seqs):
    state = stationary.copy()
    for t in range(N_CTX):
        state = state @ tmats[all_seqs_np[i, t]]
        state /= state.sum()
        all_beliefs[i, t] = state

beliefs_flat = all_beliefs.reshape(-1, n_states)
n_data = beliefs_flat.shape[0]
print(f'Sequences: {n_all_seqs:,}  |  Data points: {n_data:,}')

# ── Analysis function (paper methodology) ──
final_resid_key = f'blocks.{N_LAYERS - 1}.hook_resid_post'

def run_analysis(return_acts=False):
    model.eval()
    act_chunks = []
    with torch.no_grad():
        for s in range(0, n_all_seqs, 2048):
            e = min(s + 2048, n_all_seqs)
            _, cache = model.run_with_cache(all_seqs_torch[s:e].to(DEVICE))
            act_chunks.append(cache[final_resid_key].detach().cpu().numpy())
            del cache
    acts_flat = np.concatenate(act_chunks, axis=0).reshape(-1, D_MODEL)

    # Uniform-weight least squares (Appendix A.5)
    X = np.column_stack([np.ones(n_data), acts_flat]).astype(np.float64)
    Y = beliefs_flat.astype(np.float64)
    beta, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
    predicted = X @ beta

    residuals = predicted - Y
    paper_mse = float(np.mean(np.sum(residuals**2, axis=1)))
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((Y - Y.mean(axis=0))**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

    result = {'paper_mse': paper_mse, 'r2': r2, 'beta': beta, 'predicted': predicted, 'targets': Y}
    if return_acts:
        result['acts_flat'] = acts_flat
    return result

# ── Training loop ──
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)
rng = np.random.default_rng(SEED)

checkpoints = sorted(set(
    [0, 10_000, 50_000, 100_000, 250_000, 500_000, 750_000, NUM_STEPS]
))
checkpoints = [c for c in checkpoints if c <= NUM_STEPS]
checkpoint_results = {}
losses = []
model.train()

pbar = tqdm(range(NUM_STEPS + 1), desc='Training')
for step in pbar:
    if step in checkpoints:
        res = run_analysis()
        checkpoint_results[step] = res
        pbar.set_postfix(mse=f"{res['paper_mse']:.6f}", r2=f"{res['r2']:.4f}")
        model.train()
    if step == 0:
        continue

    inputs, labels = generate_batch(rng)
    outputs = model(inputs)
    loss = loss_fn(outputs.reshape(-1, VOCAB_SIZE), labels.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 5000 == 0:
        pbar.set_postfix(loss=f'{loss.item():.4f}')

print(f'\nFinal loss: {losses[-1]:.4f}')
print('Checkpoint MSE:')
for s in sorted(checkpoint_results):
    r = checkpoint_results[s]
    print(f"  Step {s:>9,}: MSE={r['paper_mse']:.6f}  R2={r['r2']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Training loss
ax = axes[0]
ax.plot(losses, lw=0.2, alpha=0.3, label='per-step')
w = min(2000, len(losses) // 5) if len(losses) > 10 else 1
if w > 1:
    sm = np.convolve(losses, np.ones(w)/w, mode='valid')
    ax.plot(np.arange(w-1, w-1+len(sm)), sm, lw=1.2, label=f'{w}-step avg')
ax.set_xlabel('Step'); ax.set_ylabel('Cross-Entropy Loss'); ax.set_title('Training Loss'); ax.legend()

# MSE over training (Figure 6D)
ax = axes[1]
steps = sorted(checkpoint_results)
mses = [checkpoint_results[s]['paper_mse'] for s in steps]
ax.plot(steps, mses, 'o-', ms=5)
ax.axhline(0.0004, color='red', ls='--', lw=0.8, label='Paper (1M steps): 0.0004')
ax.set_xlabel('Step'); ax.set_ylabel('MSE'); ax.set_title('Belief Geometry MSE (Fig 6D)')
ax.set_yscale('log'); ax.legend(fontsize=8)
for s, m in zip(steps, mses):
    ax.annotate(f'{m:.4f}', (s, m), textcoords='offset points', xytext=(5, 5), fontsize=7)

plt.tight_layout(); plt.show()

## 5. Activation Analysis — Reproducing Figure 5

In [ ]:
final = run_analysis(return_acts=True)
predicted = final['predicted']
gt = final['targets']

print('=== Final Analysis ===')
print(f"  Paper MSE:  {final['paper_mse']:.6f}   (paper Fig S1: ~0.0004)")
print(f"  R-squared:  {final['r2']:.6f}")
print(f'  Data pts:   {n_data:,}')

paper_ref = 0.0004
our = final['paper_mse']
if our < paper_ref * 2:
    print(f'  PASS: MSE ({our:.4f}) is close to paper reference ({paper_ref}).')
elif our < paper_ref * 5:
    print(f'  CLOSE: MSE is {our/paper_ref:.1f}x paper. More steps may help.')
else:
    print(f'  GAP: MSE is {our/paper_ref:.1f}x paper. Increase NUM_STEPS.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
plot_simplex(belief_states_gt, 'B  Ground-Truth Belief Geometry', axes[0], s=2, alpha=0.6)
plot_simplex(predicted, f"C  Residual Stream (MSE={final['paper_mse']:.4f})", axes[1],
             colors=gt, s=0.3, alpha=0.3)
fig.suptitle('Figure 5 Reproduction: Mess3 Belief State Geometry', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

## 6. Controls (Figure 6B–D)

- **Cross-validation**: 20% train / 80% test, 100 iterations (paper uses 1000).
- **Shuffle**: Randomly permute pairings, 100 iterations.

In [ ]:
acts_flat = final['acts_flat']
N = n_data
N_TRIALS = 100
ctrl_rng = np.random.default_rng(SEED)

# ── Cross-validation: 20% train, 80% test ──
cv_test, cv_train = [], []
for _ in tqdm(range(N_TRIALS), desc='Cross-validation'):
    idx = ctrl_rng.permutation(N)
    nt = int(0.2 * N)
    tr_i, te_i = idx[:nt], idx[nt:]
    X_tr = np.column_stack([np.ones(nt), acts_flat[tr_i]])
    beta, _, _, _ = np.linalg.lstsq(X_tr, beliefs_flat[tr_i], rcond=None)
    # Test
    X_te = np.column_stack([np.ones(len(te_i)), acts_flat[te_i]])
    te_resid = X_te @ beta - beliefs_flat[te_i]
    cv_test.append(float(np.mean(np.sum(te_resid**2, axis=1))))
    # Train
    tr_resid = X_tr @ beta - beliefs_flat[tr_i]
    cv_train.append(float(np.mean(np.sum(tr_resid**2, axis=1))))

# ── Shuffle ──
shuffle_mses = []
for _ in tqdm(range(N_TRIALS), desc='Shuffle'):
    perm = ctrl_rng.permutation(N)
    X = np.column_stack([np.ones(N), acts_flat])
    beta, _, _, _ = np.linalg.lstsq(X, beliefs_flat[perm], rcond=None)
    resid = X @ beta - beliefs_flat[perm]
    shuffle_mses.append(float(np.mean(np.sum(resid**2, axis=1))))

print('\n=== Control Results ===')
print(f"Full MSE:         {final['paper_mse']:.6f}   (paper: ~0.0004)")
print(f'CV test MSE:      {np.mean(cv_test):.6f} +/- {np.std(cv_test):.6f}')
print(f'CV train MSE:     {np.mean(cv_train):.6f} +/- {np.std(cv_train):.6f}')
print(f'Shuffle MSE:      {np.mean(shuffle_mses):.6f} +/- {np.std(shuffle_mses):.6f}   (paper: ~0.01)')
sh_ratio = np.mean(shuffle_mses) / max(final['paper_mse'], 1e-10)
print(f'Shuffle/Full:     {sh_ratio:.1f}x   (paper: ~25x)')

fig, ax = plt.subplots(figsize=(8, 4))
labels = ['Full', 'CV test\\n(80%)', 'CV train\\n(20%)', 'Shuffle']
means = [final['paper_mse'], np.mean(cv_test), np.mean(cv_train), np.mean(shuffle_mses)]
stds = [0, np.std(cv_test), np.std(cv_train), np.std(shuffle_mses)]
bars = ax.bar(labels, means, yerr=stds, capsize=5, color=['#2196F3','#9C27B0','#4CAF50','#F44336'], alpha=0.8)
ax.set_ylabel('MSE'); ax.set_title('Figure 6D: Regression Error Comparison')
for bar, m in zip(bars, means):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f'{m:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

---
# Part 2: RRXOR (Figure 7)

The RRXOR process has 5 states, binary vocabulary {0,1}, and 36 distinct belief
states in a 4-simplex. The paper's key finding: unlike Mess3, the belief geometry
is **not** in the final layer alone — it is spread across layers and only recovered
by **concatenating** all layers' residual streams.

In [ ]:
# ── RRXOR process definition (Appendix A.3) ──
def rrxor(p1, p2):
    """RRXOR transition matrices. 5 states: S,0,1,T,F. Vocab: {0,1}."""
    T = np.zeros((2, 5, 5), dtype=np.float64)
    S, Z, O, Tr, F = 0, 1, 2, 3, 4  # state indices
    T[0, S, Z] = p1;    T[1, S, O] = 1 - p1
    T[0, Z, F] = p2;    T[1, Z, Tr] = 1 - p2
    T[0, O, Tr] = p2;   T[1, O, F] = 1 - p2
    T[1, Tr, S] = 1.0;  T[0, F, S] = 1.0
    return T

rrxor_tmats = rrxor(0.5, 0.5)
RRXOR_VOCAB = 2
RRXOR_STATES = 5

# Verify against paper (Appendix A.3)
exp_T0 = np.array([[0,.5,0,0,0],[0,0,0,0,.5],[0,0,0,.5,0],[0,0,0,0,0],[1,0,0,0,0]])
exp_T1 = np.array([[0,0,.5,0,0],[0,0,0,.5,0],[0,0,0,0,.5],[1,0,0,0,0],[0,0,0,0,0]])
assert np.allclose(rrxor_tmats[0], exp_T0), f'T^(0) mismatch:\n{rrxor_tmats[0]}'
assert np.allclose(rrxor_tmats[1], exp_T1), f'T^(1) mismatch:\n{rrxor_tmats[1]}'
print('PASS: RRXOR matrices match paper.')

# Stationary distribution
T_net = rrxor_tmats.sum(axis=0)
eigvals, eigvecs = np.linalg.eig(T_net.T)
rrxor_stationary = eigvecs[:, np.isclose(eigvals, 1)].real.squeeze()
rrxor_stationary = rrxor_stationary / rrxor_stationary.sum()
print(f'Stationary: {rrxor_stationary}')

# ── Init RRXOR transformer (same architecture, d_vocab=2) ──
torch.manual_seed(SEED)
rrxor_n_heads = N_HEADS
try:
    rrxor_cfg = HookedTransformerConfig(
        n_ctx=N_CTX, d_model=D_MODEL, d_head=D_HEAD, n_heads=rrxor_n_heads,
        n_layers=N_LAYERS, d_mlp=D_MLP, d_vocab=RRXOR_VOCAB,
        act_fn='relu', normalization_type='LN', device=DEVICE, seed=SEED,
    )
    rrxor_model = HookedTransformer(rrxor_cfg)
except Exception:
    rrxor_n_heads = 8
    rrxor_cfg = HookedTransformerConfig(
        n_ctx=N_CTX, d_model=D_MODEL, d_head=D_HEAD, n_heads=rrxor_n_heads,
        n_layers=N_LAYERS, d_mlp=D_MLP, d_vocab=RRXOR_VOCAB,
        act_fn='relu', normalization_type='LN', device=DEVICE, seed=SEED,
    )
    rrxor_model = HookedTransformer(rrxor_cfg)
print(f'RRXOR model: {sum(p.numel() for p in rrxor_model.parameters()):,} params')

# ── Precompute exhaustive RRXOR data ──
print('Enumerating RRXOR sequences...')
rrxor_all_seqs_np = np.array(list(itertools.product(range(RRXOR_VOCAB), repeat=N_CTX)), dtype=np.int32)
rrxor_all_seqs_torch = torch.tensor(rrxor_all_seqs_np, dtype=torch.long)
rrxor_n_all = rrxor_all_seqs_np.shape[0]

print('Computing RRXOR belief states...')
rrxor_all_beliefs = np.zeros((rrxor_n_all, N_CTX, RRXOR_STATES), dtype=np.float32)
for i in range(rrxor_n_all):
    state = rrxor_stationary.copy()
    for t in range(N_CTX):
        state = state @ rrxor_tmats[rrxor_all_seqs_np[i, t]]
        s = state.sum()
        state = state / s if s > 0 else state
        rrxor_all_beliefs[i, t] = state

rrxor_beliefs_flat = rrxor_all_beliefs.reshape(-1, RRXOR_STATES)
rrxor_n_data = rrxor_beliefs_flat.shape[0]
print(f'RRXOR sequences: {rrxor_n_all:,}  |  Data points: {rrxor_n_data:,}')

# ── RRXOR data generation ──
def rrxor_generate_batch(rng_obj):
    tokens = np.zeros((BATCH_SIZE, SEQUENCE_LEN), dtype=np.int64)
    for i in range(BATCH_SIZE):
        state = rrxor_stationary.copy()
        for t in range(SEQUENCE_LEN):
            obs_probs = np.array([np.sum(state @ rrxor_tmats[x]) for x in range(RRXOR_VOCAB)])
            obs_probs = np.maximum(obs_probs, 0)
            s = obs_probs.sum()
            obs_probs = obs_probs / s if s > 0 else np.ones(RRXOR_VOCAB) / RRXOR_VOCAB
            tok = rng_obj.choice(RRXOR_VOCAB, p=obs_probs)
            tokens[i, t] = tok
            state = state @ rrxor_tmats[tok]
            s = state.sum()
            state = state / s if s > 0 else state
    return torch.tensor(tokens[:, :-1], device=DEVICE), \
           torch.tensor(tokens[:, 1:], device=DEVICE, dtype=torch.long)

# ── Train RRXOR ──
rrxor_loss_fn = torch.nn.CrossEntropyLoss()
rrxor_optimizer = torch.optim.SGD(rrxor_model.parameters(), lr=LEARNING_RATE)
rrxor_rng = np.random.default_rng(SEED + 1000)

rrxor_checkpoints = sorted(set(
    [0, 10_000, 50_000, 100_000, 250_000, 500_000, 750_000, NUM_STEPS]
))
rrxor_checkpoints = [c for c in rrxor_checkpoints if c <= NUM_STEPS]
rrxor_checkpoint_results = {}
rrxor_losses = []
rrxor_model.train()

# Quick analysis function for RRXOR checkpoints (final layer only for speed)
def rrxor_quick_analysis():
    rrxor_model.eval()
    act_ch = []
    fk = f'blocks.{N_LAYERS-1}.hook_resid_post'
    with torch.no_grad():
        for s in range(0, rrxor_n_all, 2048):
            e = min(s + 2048, rrxor_n_all)
            _, cache = rrxor_model.run_with_cache(rrxor_all_seqs_torch[s:e].to(DEVICE))
            act_ch.append(cache[fk].detach().cpu().numpy())
            del cache
    af = np.concatenate(act_ch, axis=0).reshape(-1, D_MODEL)
    X = np.column_stack([np.ones(rrxor_n_data), af]).astype(np.float64)
    Y = rrxor_beliefs_flat.astype(np.float64)
    beta, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
    resid = X @ beta - Y
    mse = float(np.mean(np.sum(resid**2, axis=1)))
    ss_res = np.sum(resid**2); ss_tot = np.sum((Y - Y.mean(axis=0))**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return {'paper_mse': mse, 'r2': r2}

pbar = tqdm(range(NUM_STEPS + 1), desc='Training RRXOR')
for step in pbar:
    if step in rrxor_checkpoints:
        sc = rrxor_quick_analysis()
        rrxor_checkpoint_results[step] = sc
        pbar.set_postfix(mse=f"{sc['paper_mse']:.6f}", r2=f"{sc['r2']:.4f}")
        rrxor_model.train()
    if step == 0:
        continue
    inp, lab = rrxor_generate_batch(rrxor_rng)
    out = rrxor_model(inp)
    loss = rrxor_loss_fn(out.reshape(-1, RRXOR_VOCAB), lab.reshape(-1))
    rrxor_optimizer.zero_grad(); loss.backward(); rrxor_optimizer.step()
    rrxor_losses.append(loss.item())
    if step % 5000 == 0:
        pbar.set_postfix(loss=f'{loss.item():.4f}')

print(f'\nRRXOR final loss: {rrxor_losses[-1]:.4f}')
for s in sorted(rrxor_checkpoint_results):
    r = rrxor_checkpoint_results[s]
    print(f"  Step {s:>9,}: MSE={r['paper_mse']:.6f}  R2={r['r2']:.4f}")

In [ ]:
# ── RRXOR: Figure 7 analysis ──
# The paper's key finding: RRXOR belief geometry is NOT in the final layer,
# but IS in the concatenation of all layers.

rrxor_final_key = f'blocks.{N_LAYERS - 1}.hook_resid_post'
rrxor_all_layer_keys = [f'blocks.{l}.hook_resid_post' for l in range(N_LAYERS)]

def rrxor_run_analysis(return_acts=False):
    rrxor_model.eval()
    # Collect activations from ALL layers (not just final)
    layer_acts = {k: [] for k in rrxor_all_layer_keys}
    with torch.no_grad():
        for s in range(0, rrxor_n_all, 2048):
            e = min(s + 2048, rrxor_n_all)
            _, cache = rrxor_model.run_with_cache(rrxor_all_seqs_torch[s:e].to(DEVICE))
            for k in rrxor_all_layer_keys:
                layer_acts[k].append(cache[k].detach().cpu().numpy())
            del cache
    layer_acts = {k: np.concatenate(v, axis=0).reshape(-1, D_MODEL) for k, v in layer_acts.items()}

    Y = rrxor_beliefs_flat.astype(np.float64)
    results = {}

    # Per-layer regression
    for k in rrxor_all_layer_keys:
        X = np.column_stack([np.ones(rrxor_n_data), layer_acts[k]]).astype(np.float64)
        beta, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
        resid = X @ beta - Y
        mse = float(np.mean(np.sum(resid**2, axis=1)))
        ss_res = np.sum(resid**2); ss_tot = np.sum((Y - Y.mean(axis=0))**2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        results[k] = {'paper_mse': mse, 'r2': r2}

    # Concatenated regression (all layers)
    concat_acts = np.concatenate([layer_acts[k] for k in rrxor_all_layer_keys], axis=1)
    X = np.column_stack([np.ones(rrxor_n_data), concat_acts]).astype(np.float64)
    beta, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
    predicted = X @ beta
    resid = predicted - Y
    mse = float(np.mean(np.sum(resid**2, axis=1)))
    ss_res = np.sum(resid**2); ss_tot = np.sum((Y - Y.mean(axis=0))**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    results['concatenated'] = {'paper_mse': mse, 'r2': r2, 'predicted': predicted, 'targets': Y}

    out = {'per_layer': results, 'paper_mse': results['concatenated']['paper_mse'],
           'r2': results['concatenated']['r2']}
    if return_acts:
        out['concat_acts'] = concat_acts
    return out

rrxor_final = rrxor_run_analysis(return_acts=True)

print('=== RRXOR: Per-Layer vs Concatenated MSE (Figure 7E) ===')
for k in rrxor_all_layer_keys:
    r = rrxor_final['per_layer'][k]
    label = k.replace('blocks.', 'L').replace('.hook_resid_post', '')
    print(f"  {label}: MSE={r['paper_mse']:.6f}  R2={r['r2']:.4f}")
r = rrxor_final['per_layer']['concatenated']
print(f"  Concat: MSE={r['paper_mse']:.6f}  R2={r['r2']:.4f}")
print(f'\nPaper finding: final layer MSE >> concat MSE (geometry spread across layers)')

# ── Figure 7E: bar chart ──
fig, ax = plt.subplots(figsize=(8, 4))
bar_labels = [f'L{l}' for l in range(N_LAYERS)] + ['Concat']
bar_mses = [rrxor_final['per_layer'][k]['paper_mse'] for k in rrxor_all_layer_keys]
bar_mses.append(rrxor_final['per_layer']['concatenated']['paper_mse'])
colors_b = ['#90CAF9'] * N_LAYERS + ['#2196F3']
ax.bar(bar_labels, bar_mses, color=colors_b, alpha=0.8)
ax.set_ylabel('MSE'); ax.set_title('RRXOR: Per-Layer vs Concatenated MSE (Figure 7E)')
for i, m in enumerate(bar_mses):
    ax.text(i, m, f'{m:.4f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.show()

---
# Part 3: Save Checkpoints

Download both trained models as a single zip. Unzip into
`notebooks/checkpoints/` in your repo:
```
notebooks/checkpoints/
├── mess3_checkpoint/
│   ├── model_weights.pt
│   ├── config.json
│   └── training_history.npz
└── rrxor_checkpoint/
    ├── model_weights.pt
    ├── config.json
    └── training_history.npz
```

In [ ]:
# ── Save both model checkpoints ──
import json, zipfile, io, os

def save_checkpoint(save_dir, model_obj, config_dict, loss_list, ckpt_results_dict):
    os.makedirs(save_dir, exist_ok=True)
    torch.save(model_obj.state_dict(), f'{save_dir}/model_weights.pt')
    with open(f'{save_dir}/config.json', 'w') as f:
        json.dump(config_dict, f, indent=2)
    np.savez_compressed(f'{save_dir}/training_history.npz',
        losses=np.array(loss_list),
        checkpoint_steps=np.array(sorted(ckpt_results_dict.keys())),
        checkpoint_mses=np.array([ckpt_results_dict[s]['paper_mse'] for s in sorted(ckpt_results_dict)]),
        checkpoint_r2s=np.array([ckpt_results_dict[s]['r2'] for s in sorted(ckpt_results_dict)]),
    )
    print(f'  Saved to {save_dir}/')
    for fn in sorted(os.listdir(save_dir)):
        sz = os.path.getsize(f'{save_dir}/{fn}')
        print(f'    {fn}: {sz/1024:.0f} KB')

# Save Mess3
print('=== Saving Mess3 checkpoint ===')
save_checkpoint('mess3_checkpoint', model, {
    'process': 'mess3', 'mess3_x': MESS3_X, 'mess3_a': MESS3_A,
    'n_ctx': N_CTX, 'd_model': D_MODEL, 'd_head': D_HEAD, 'n_heads': N_HEADS,
    'n_layers': N_LAYERS, 'd_mlp': D_MLP, 'd_vocab': VOCAB_SIZE,
    'act_fn': 'relu', 'normalization_type': 'LN', 'seed': SEED,
    'num_steps_trained': NUM_STEPS, 'final_loss': float(losses[-1]),
    'final_paper_mse': float(final['paper_mse']), 'final_r2': float(final['r2']),
}, losses, checkpoint_results)

# Save RRXOR
print('\n=== Saving RRXOR checkpoint ===')
save_checkpoint('rrxor_checkpoint', rrxor_model, {
    'process': 'rrxor', 'p1': 0.5, 'p2': 0.5,
    'n_ctx': N_CTX, 'd_model': D_MODEL, 'd_head': D_HEAD, 'n_heads': rrxor_n_heads,
    'n_layers': N_LAYERS, 'd_mlp': D_MLP, 'd_vocab': 2,
    'act_fn': 'relu', 'normalization_type': 'LN', 'seed': SEED,
    'num_steps_trained': NUM_STEPS, 'final_loss': float(rrxor_losses[-1]),
    'final_paper_mse': float(rrxor_final['paper_mse']), 'final_r2': float(rrxor_final['r2']),
}, rrxor_losses, rrxor_checkpoint_results)

# Zip and download
print('\n=== Packaging for download ===')
with zipfile.ZipFile('belief_state_checkpoints.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files_list in os.walk('mess3_checkpoint'):
        for fn in files_list:
            fp = os.path.join(root, fn)
            zf.write(fp)
    for root, _, files_list in os.walk('rrxor_checkpoint'):
        for fn in files_list:
            fp = os.path.join(root, fn)
            zf.write(fp)
sz = os.path.getsize('belief_state_checkpoints.zip')
print(f'belief_state_checkpoints.zip: {sz/1024:.0f} KB')

try:
    from google.colab import files
    files.download('belief_state_checkpoints.zip')
    print('Download started.')
except ImportError:
    print('Not in Colab — zip file saved locally.')